In [ ]:
import deeptime as dpt
from tqdm.notebook import tqdm # for progress bar
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pyemma as pem
import pickle
import os

In [ ]:
def extract_msm_pcca_for_subgroup(discrete_trajs, state_set, lagtime):
    """
    Extract and build a Markov State Model (MSM) for a specific subgroup of microstates.
    
    This function processes discrete trajectories to create an MSM for a subset of states
    (subgroup) that are connected within the full state space. It filters trajectories to
    only include states from the specified subgroup, relabels them to 0-based indexing,
    and constructs an MSM using the deeptime library.
    
    Parameters
    ----------
    discrete_trajs : list of numpy.ndarray
        List of discrete trajectory arrays, where each trajectory contains microstate
        indices. Each trajectory should be a 1D numpy array of integers representing
        the microstate assignments from clustering (e.g., K-means clustering of
        molecular dynamics data).
        
    state_set : set or list
        Set of microstate indices that define the subgroup to analyze. These should
        be the global microstate indices from the original clustering.
        
    lagtime : int
        Lag time (in trajectory steps) for constructing the MSM transition matrix.
        This parameter determines the time resolution of the Markov model.
        
    Returns
    -------
        -------
    msm : deeptime.markov.msm.MaximumLikelihoodMSM
        Fitted Markov State Model object for the subgroup. This object contains
        the transition matrix, stationary distribution, and other MSM properties.
        
    state_map : dict
        Mapping from global microstate indices to subgroup microstate indices
        (0-based). Keys are global microstate indices, values are subgroup indices.
        Example: {32: 0, 37: 1, 29: 2, ...}
        
    state_map_reverse : dict
        Reverse mapping from subgroup microstate indices to global microstate indices.
        Keys are subgroup indices (0-based), values are global microstate indices.
        Example: {0: 32, 1: 37, 2: 29, ...}
        
    Notes
    -----
    - The function filters out trajectories that contain fewer than 2 states from
      the specified subgroup to ensure meaningful MSM construction.
    - Microstates within the subgroup are relabeled to 0-based indexing for the
      MSM construction, but the mapping dictionaries preserve the relationship
      to the original global microstate indices.
    - This function is typically used as part of a larger workflow for handling
      disconnected state spaces in molecular dynamics simulations, where the
      full state space contains multiple disconnected components.
      
    Examples
    --------
        >>> # Assuming you have discrete trajectories and identified subgroups
    >>> discrete_trajectories = [np.array([0, 1, 2, 1, 0]), np.array([2, 3, 2, 1])]
    >>> subgroup_states = {1, 2, 3}  # States to analyze
    >>> lagtime = 1
    >>> 
    >>> msm, state_map, state_map_reverse = extract_msm_pcca_for_subgroup(
    ...     discrete_trajectories, subgroup_states, lagtime
    ... )
    >>> 
    >>> print(f"MSM transition matrix shape: {msm.transition_matrix.shape}")
    >>> print(f"State mapping: {state_map}")
    >>> print(f"Reverse mapping: {state_map_reverse}")
    
    See Also
    --------
    MSM_disconnected_sets : Function that uses this to process multiple subgroups
    deeptime.markov.msm.MaximumLikelihoodMSM : The MSM implementation used
    """
    # Filter trajectories
    filtered_trajs = []
    for traj in discrete_trajs:
        filtered = [s for s in traj if s in state_set]
        # what if filtered has length of 1?
        if len(filtered) > 1:
            filtered_trajs.append(filtered)

    # Relabel to 0-based
    #global microstate -> subgroup miscrostate (subgroup microstate is from 0->len(state_set))
    state_map = {s: i for i, s in enumerate(sorted(state_set))}
    #subgroup microstate -> global
    state_map_reverse = {i:s for i, s in enumerate(sorted(state_set))}
    relabeled_trajs = [np.array([state_map[s] for s in traj]) for traj in filtered_trajs]

    # Build MSM
    msm = dpt.markov.msm.MaximumLikelihoodMSM(lagtime=lagtime).fit(relabeled_trajs).fetch_model()

    return msm, state_map, state_map_reverse

In [ ]:
def MSM_disconnected_sets(sub_groups, n_states):
    """
    Process multiple disconnected subgroups to identify macrostates using MSM and PCCA analysis.
    
    This function builds Markov State Models (MSMs) for each subgroup individually, performs 
    Perron Cluster Cluster Analysis (PCCA) to identify macrostates, and consolidates the 
    results into a global framework. Essential for handling disconnected state spaces where 
    the full conformational landscape contains multiple isolated regions.
    
    Parameters
    ----------
    sub_groups : list of sets
        List of disconnected subgroups, where each subgroup is a set of microstate indices.
        Each set contains global microstate indices from the original clustering.
        Typically obtained from dpt.markov.tools.estimation.connected_sets().
        Example: [{0, 1, 2, 3, 4}, {5, 6, 7, 8, 9}] for two subgroups
        
    n_states : list of int
        Number of macrostates to identify for each subgroup.
        Must equal the number of subgroups in sub_groups.
        Example: [2, 3] means 2 macrostates for subgroup 0, 3 for subgroup 1
        Constraint: len(n_states) == len(sub_groups)
    
    Returns
    -------
    global_pcca_sets : list of lists
        Global macrostate assignments, where each inner list contains global microstate 
        indices belonging to one macrostate. Macrostates are numbered globally across 
        all subgroups.
        Format: [[microstates_in_macrostate_0], [microstates_in_macrostate_1], ...]
        Example: [[2, 3, 5, 6, 9, ...], [0, 8, 13, 18, ...], ...]
        
    global_most_prob_microstate : list of int
        Global microstate indices representing the most probable microstate for each 
        macrostate. Used for identifying representative structures for each macrostate.
        Format: [most_prob_microstate_for_macrostate_0, most_prob_microstate_for_macrostate_1, ...]
        Example: [32, 37, 29, 88, 68]
        
    Notes
    -----
    - The function assumes len(n_states) == len(sub_groups)
    - All returned indices refer to the original global microstate numbering
    - Macrostates are numbered sequentially across all subgroups
    - Requires discrete_trajectories and lagtime to be defined globally
    - Uses deeptime's PCCA implementation for macrostate identification
    
    Algorithm
    ---------
    1. Subgroup Processing Loop: Iterates through each disconnected subgroup
    2. MSM Construction: Builds MSM for each subgroup using extract_msm_pcca_for_subgroup()
    3. PCCA Analysis: Performs Perron Cluster Cluster Analysis on each subgroup's MSM
    4. Global Consolidation: Converts local assignments to global microstate indices
    5. Output Generation: Prints detailed info and returns consolidated assignments
        Examples
    --------
    >>> # Identify disconnected components first
    >>> c_matrix = dpt.markov.tools.estimation.count_matrix(discrete_trajectories, lagtime).toarray()
    >>> sub_groups = dpt.markov.tools.estimation.connected_sets(c_matrix)
    >>> 
    >>> # Process subgroups to identify macrostates
    >>> global_pcca_sets, global_most_prob_microstate = MSM_disconnected_sets(
    ...     sub_groups, n_states=[2, 3]
    ... )
    >>> 
    >>> # Access results
    >>> print(f"Total macrostates: {len(global_pcca_sets)}")
    >>> print(f"Macrostate 0 microstates: {global_pcca_sets[0]}")
    >>> print(f"Most probable microstate for macrostate 0: {global_most_prob_microstate[0]}")
    
    Output Format
    ------------
    The function prints detailed information for each identified macrostate:
    [Macrostate 0] Subgroup 0, original macrostate 0: size=41, most prob. microstate=32
    [Macrostate 1] Subgroup 0, original macrostate 1: size=19, most prob. microstate=37
    [Macrostate 2] Subgroup 1, original macrostate 0: size=17, most prob. microstate=29
    
    See Also
    --------
    extract_msm_pcca_for_subgroup : Core function for processing individual subgroups
    dpt.markov.tools.estimation.connected_sets : Identifies disconnected components
    dpt.markov.tools.estimation.count_matrix : Creates count matrix for connectivity analysis
    """
    # check If number of elements in n_states must be equal to number of sub_groups
    if len(n_states) != len(sub_groups):
        raise ValueError(
            f"Number of elements in n_states ({len(n_states)}) must equal "
            f"the number of sub_groups ({len(sub_groups)})"
        )
    
    # Store the data across all subgroups as flat lists
    global_pcca_sets = []  # Each item is a list of global microstates in one macrostate
    global_most_prob_microstate = []  # Each item is a global microstate index
    msm_list = []
    
    
    # Loop through each connected component
    for i, subgroup in enumerate(sub_groups):
        try:
            msm, state_map, state_map_reverse = extract_msm_pcca_for_subgroup(discrete_trajectories, subgroup, lagtime=1)
            print(f"MSM estimation for subgroup {i}:")
            print("Top 10 eigenvalues and timescales:")
            print("Top 10 eigenvalues:", msm.eigenvalues()[:10])
            print("Top 10 relaxation timescales:", msm.timescales()[:10])

            # append msm to msm_list
            msm_list.append(msm)
            n_macrostates = n_states[i]
            pcca = msm.pcca(n_macrostates)
            
            # Find most probable microstate per macrostate (local indices)
            most_prob_microstate_local = np.argmax(pcca.memberships, axis=0)
            
            # Convert PCCA sets and most_prob_microstate to global microstate indices
            for j in range(n_macrostates):
                macrostate_global = [state_map_reverse[m] for m in pcca.sets[j]]
                most_prob_global = state_map_reverse[most_prob_microstate_local[j]]
        
                global_pcca_sets.append(macrostate_global)  # 1 macrostate per append
                global_most_prob_microstate.append(most_prob_global)
        
                print(f"[Macrostate {len(global_pcca_sets)-1}] Subgroup {i}, original macrostate {j}: "
                    f"size={len(macrostate_global)}, most prob. microstate={most_prob_global}")
            
            print("-----")
        except Exception as e:
            raise RuntimeError(f"Error processing subgroup {i}: {e}")

    return global_pcca_sets, global_most_prob_microstate, msm_list

In [ ]:
# Step 1: Load your Q-G data
data = np.load("QG.npy")  # shape (50, 13333, 2)

# Step 2: Standardize the data
scaler = StandardScaler()
raw_data_all = np.concatenate(data, axis=0)
scaler.fit(raw_data_all)
processed_data = [scaler.transform(traj) for traj in data]  # List of 2D arrays
conc_processed_data = np.concatenate(processed_data, axis=0)

In [ ]:
# Use KMeans from deeptime 
n_clusters = 100
kmeans_model = dpt.clustering.KMeans(n_clusters=n_clusters,
                                    max_iter=500,
                                    fixed_seed=True,
                                    progress=tqdm
                                    ).fit_fetch(processed_data)

# Transform each trajectory to discrete state trajectory
discrete_trajectories = [kmeans_model.transform(traj) for traj in processed_data]

# Get cluster centers (in scaled space)
cluster_centers_scaled = kmeans_model.cluster_centers

# Inverse transform to original Q-G scale
cluster_centers_original = scaler.inverse_transform(cluster_centers_scaled)

# Optional: Check shapes and values
print("Number of trajectories:", len(discrete_trajectories))
print("Shape of first discrete trajectory:", discrete_trajectories[0].shape)
print("Unique states in first trajectory:", np.unique(discrete_trajectories[0]))

The following will handle the situation multiple disconnected subgroups in the model.
Before execute this code, need to check if the normal MSM model ultilize the full microstates. If yes, just process with the normal compacted code.
Else, execute this code to work on each subgroup individually.

In [ ]:
# get the count_matrix of states
lagtime = 1 
c_matrix = dpt.markov.tools.estimation.count_matrix(discrete_trajectories, lagtime).toarray()
sub_groups = dpt.markov.tools.estimation.connected_sets(c_matrix)
print(f"Number of sub-groups: {len(sub_groups)}")
for idx, subgroup in enumerate(sub_groups):
    print("------")
    print(f"Sub-group: {idx}, size={len(subgroup)}")
    print("Element of microstates in this subgroup:")
    print(subgroup)
    
# print(sub_groups)

In [ ]:
# Concatenate everything
# Plot the free energy landscape of each subgroup to get a sense of how many macrostates each subgroup may have
flat_discrete = np.concatenate(discrete_trajectories, axis=0)

n_panels = len(sub_groups)
total_plots = n_panels + 1
ncols = 2
nrows = int(np.ceil(total_plots / ncols))

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(10, 3 * nrows))
axes = axes.flatten()  # Flatten in case of single row

# Plot each connected set
for i in range(n_panels):
    subgroup = set(sub_groups[i])
    mask = np.isin(flat_discrete, list(subgroup))
    subgroup_data_qg = raw_data_all[mask]

    ax = axes[i]
    pem.plots.plot_free_energy(subgroup_data_qg[:, 0], subgroup_data_qg[:, 1], nbins=30, ax=ax)
    ax.set_ylim([0, np.max(subgroup_data_qg[:, 1])])
    ax.set_xlim([0, 1])
    ax.set_title(f"Subgroup {i}")

# Plot the global landscape
ax = axes[n_panels]
pem.plots.plot_free_energy(raw_data_all[:, 0], raw_data_all[:, 1], nbins=50, ax=ax)
ax.set_ylim([0, np.max(raw_data_all[:, 1])])
ax.set_xlim([0, 1])
ax.set_title("Global")

# Hide any unused subplots
for j in range(n_panels + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
global_pcca_sets, global_most_prob_microstate, msm_list = MSM_disconnected_sets(sub_groups, n_states=[2,3])

In [ ]:
# un-sorted macrostate assignment
macrostate_assignment = np.empty(n_clusters, dtype=int)
for idx, macro_set in enumerate(global_pcca_sets):
    print(idx, macro_set)
    for micro_state in macro_set:
        macrostate_assignment[micro_state] = idx

In [ ]:
# Sort the macrostates by the Q value of the most probable microstate for more intuitive macrostate assignment
print("Original macrostates and their [Q, G] values:")
for macro_idx, micro_idx in enumerate(global_most_prob_microstate):
    print(f"Macrostate {macro_idx}: {micro_idx}: {cluster_centers_original[micro_idx]}")
# because the macrostate indices from deeptime are not sorted for intutitive, we need to sort them. so that the smallest macrostate index is the most stable macrostate.
# corresponding to the state with the lowest Q.
print("------------")
print("Sort the macrostates by their Q value and the mapping:")
macro_state_sorted = np.argsort(cluster_centers_original[global_most_prob_microstate][:,0])
macro_state_mapping = {}
for i in range(len(macro_state_sorted)):
    print(f"Macrostate old: {macro_state_sorted[i]} -> New: {i}")
    macro_state_mapping[macro_state_sorted[i]] = i

In [ ]:
# convert macrostate_assignment to the sorted macrostate_assignment
macrostate_assignment_sorted = np.zeros_like(macrostate_assignment)
for i in range(len(macrostate_assignment)):
    macrostate_assignment_sorted[i] = macro_state_mapping[macrostate_assignment[i]]

# convert the microstates to macrostates
meta_dtrajs = [[macrostate_assignment_sorted[i] for i in dtraj] for dtraj in discrete_trajectories]
meta_dtrajs_flat = np.array(np.array(meta_dtrajs).flatten(), dtype=int)

In [ ]:
# Plot the free energy surface and the state map
x = np.array(raw_data_all[:,0])
y = np.array(raw_data_all[:,1])

nbins = 50 # number of bins for the free energy surface and the state map- Protein-specific

# Create combined figure with two panels using GridSpec
fig = plt.figure(figsize=(13, 7))
gw = int(np.floor(0.5 + 500 * fig.get_figwidth()))
gh = int(np.floor(0.5 + 500 * fig.get_figheight()))
gs = plt.GridSpec(gh, gw)
gs.update(hspace=0.0, wspace=0.0, left=0.0, right=1.0, bottom=0.0, top=1.0)
ax_box = fig.add_subplot(gs[:, :])
ax_box.set_axis_off()

# Panel A: Free Energy Surface (left)
ax_fe = fig.add_subplot(gs[500:2750, 500:3000])
# Plot on the raw data
_, _, misc = pem.plots.plot_free_energy(x, y, ax=ax_fe, nbins=nbins,
                                        cax=fig.add_subplot(gs[300:400, 500:3000]),
                                        cbar_orientation='horizontal',
                                        cmap='coolwarm',
                                        levels=100,
                                        legacy=False
                                        )

# # if need, plot the cluster centers
# ax_fe.scatter(
#        cluster_centers_original[:, 0],  # Q
#        cluster_centers_original[:, 1],  # G
#        c='k', marker='o', s=40, label='Cluster Centers'
#    )

misc['cbar'].ax.xaxis.set_ticks_position('top')
misc['cbar'].ax.xaxis.set_label_position('top')
misc['cbar'].set_label(r'-ln(P) / $\mathrm{k}_\mathrm{B}T$')

ax_fe.set_ylabel('$G$', fontsize=16)
ax_fe.set_xlabel('$Q$', fontsize=16)
ax_fe.set_ylim([np.min(y), np.max(y)])

# Panel B: State Map (right)
ax_state = fig.add_subplot(gs[500:2750, 3500:6000])

_, _, misc = pem.plots.plot_state_map(x, y, meta_dtrajs_flat, ax=ax_state, nbins=nbins,
                                      cax=fig.add_subplot(gs[300:400, 3500:6000]),
                                      cbar_orientation='horizontal'
                                      )



misc['cbar'].ax.xaxis.set_ticks_position('top')
misc['cbar'].ax.xaxis.set_label_position('top')
misc['cbar'].set_label('Macrostate')


ax_state.set_ylabel('$G$', fontsize=16)
ax_state.set_xlabel('$Q$', fontsize=16)
ax_state.set_ylim([np.min(y), np.max(y)])
# ax_state.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Add panel labels
ax_fe.text(0.05, 0.95, 'A', transform=ax_fe.transAxes, fontsize=20, 
           fontweight='bold', verticalalignment='top')
ax_state.text(0.05, 0.95, 'B', transform=ax_state.transAxes, fontsize=20, 
              fontweight='bold', verticalalignment='top')

# plt.tight_layout()
plt.savefig('QG_MSM.png', dpi=600)
plt.show()

In [ ]:
# Check state probability along the whole trajectory
n_macrostates = len(global_pcca_sets)
for i in range(n_macrostates):
    print(f"Macrostate {i}: {len(meta_dtrajs_flat[meta_dtrajs_flat==i])/len(meta_dtrajs_flat):.3f}")

# Last 200ns of each trajectory

In [ ]:
# Check state probability along the last 200ns of the trajectory
meta_dtrajs_all = np.asarray(meta_dtrajs)
meta_dtrajs_last_200ns = meta_dtrajs_all[:,-2666:]
meta_dtrajs_last_200ns_flat = np.concatenate(meta_dtrajs_last_200ns)
for i in range(n_macrostates):
    print(f"Macrostate {i}: {len(meta_dtrajs_last_200ns_flat[meta_dtrajs_last_200ns_flat==i])/len(meta_dtrajs_last_200ns_flat):.3f}")

In [ ]:
# Plot on the last 200 ns
# Plot the free energy surface and the state map
raw_data_all_last200ns = np.concatenate(data[:,-2666:, :], axis=0)

x = np.array(raw_data_all_last200ns[:,0])
y = np.array(raw_data_all_last200ns[:,1])


nbins = 50 # number of bins for the free energy surface and the state map- Protein-specific

# Create combined figure with two panels using GridSpec
fig = plt.figure(figsize=(13, 7))
gw = int(np.floor(0.5 + 500 * fig.get_figwidth()))
gh = int(np.floor(0.5 + 500 * fig.get_figheight()))
gs = plt.GridSpec(gh, gw)
gs.update(hspace=0.0, wspace=0.0, left=0.0, right=1.0, bottom=0.0, top=1.0)
ax_box = fig.add_subplot(gs[:, :])
ax_box.set_axis_off()

# Panel A: Free Energy Surface (left)
ax_fe = fig.add_subplot(gs[500:2750, 500:3000])
# Plot on the raw data
_, _, misc = pem.plots.plot_free_energy(x, y, ax=ax_fe, nbins=nbins,
                                        cax=fig.add_subplot(gs[300:400, 500:3000]),
                                        cbar_orientation='horizontal',
                                        cmap='coolwarm',
                                        levels=100,
                                        legacy=False
                                        )

# # if need, plot the cluster centers
# ax_fe.scatter(
#        cluster_centers_original[:, 0],  # Q
#        cluster_centers_original[:, 1],  # G
#        c='k', marker='o', s=40, label='Cluster Centers'
#    )

misc['cbar'].ax.xaxis.set_ticks_position('top')
misc['cbar'].ax.xaxis.set_label_position('top')
misc['cbar'].set_label(r'-ln(P) / $\mathrm{k}_\mathrm{B}T$')

ax_fe.set_ylabel('$G$', fontsize=16)
ax_fe.set_xlabel('$Q$', fontsize=16)
ax_fe.set_ylim([np.min(y), np.max(y)])

# Panel B: State Map (right)
ax_state = fig.add_subplot(gs[500:2750, 3500:6000])

_, _, misc = pem.plots.plot_state_map(x, y, meta_dtrajs_last_200ns_flat, ax=ax_state, nbins=nbins,
                                      cax=fig.add_subplot(gs[300:400, 3500:6000]),
                                      cbar_orientation='horizontal'
                                      )



misc['cbar'].ax.xaxis.set_ticks_position('top')
misc['cbar'].ax.xaxis.set_label_position('top')
misc['cbar'].set_label('Macrostate')


ax_state.set_ylabel('$G$', fontsize=16)
ax_state.set_xlabel('$Q$', fontsize=16)
ax_state.set_ylim([np.min(y), np.max(y)])
# ax_state.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Add panel labels
ax_fe.text(0.05, 0.95, 'A', transform=ax_fe.transAxes, fontsize=20, 
           fontweight='bold', verticalalignment='top')
ax_state.text(0.05, 0.95, 'B', transform=ax_state.transAxes, fontsize=20, 
              fontweight='bold', verticalalignment='top')

# plt.tight_layout()
plt.savefig('QG_MSM_last200ns.png', dpi=600)
plt.show()

In [ ]:
# Check if 'msm_results' folder exists, create if not
results_dir = 'msm_results'
os.makedirs(results_dir, exist_ok=True)

# Save arrays
np.savez(os.path.join(results_dir, 'msm_analysis_results.npz'),
         cluster_centers_scaled=cluster_centers_scaled,
         cluster_centers_original=cluster_centers_original,
         discrete_trajectories=discrete_trajectories,
         macrostate_assignment=macrostate_assignment,
         macrostate_assignment_sorted=macrostate_assignment_sorted,
         # membership=membership,
         processed_data=processed_data,
         meta_dtrajs=meta_dtrajs
)

if scaler is not None:
    with open(os.path.join(results_dir, 'scaler.pkl'), 'wb') as f:
        pickle.dump(scaler, f)

if kmeans_model is not None:
    with open(os.path.join(results_dir, 'kmeans_model.pkl'), 'wb') as f:
        pickle.dump(kmeans_model, f)
